# Cohort matching
This notebook performs propensity score matching to create balanced cohorts for depression subtyping analysis. It uses predefined covariates to match individuals with depression (F32 ICD-10 major depressive disorder) to control subjects without depression (and actually without any known ICD-10 codes), aiding comparability between groups. The steps include loading the initial cohort, calculating propensity scores, performing the matching process, and saving the combined cohort for further analysis. Additionally, subject IDs with suffixes for neuroimaging data extraction (see `02_workstation_pull.ipynb`) are extracted and saved for the control cohort. This step has not been performed in the previous notebook because those initial controls were not matched to the depression cohort. 

## 1) Imports

In [ ]:
import os
import sys
import pandas as pd

PROJECT_ROOT = '.../subtyping_depression'
COHORT_DIR = os.path.join(PROJECT_ROOT, 'source_code', 'cohort_definition')
if COHORT_DIR not in sys.path:
    sys.path.insert(0, COHORT_DIR)

from cohort_matching_utils import (
    propensity_score_matching,
    assess_covariate_balance,
    plot_propensity_distributions,
    combine_matched_cohorts,
)

from cohort_selection_utils import extract_subject_ids

%matplotlib inline

## 2) Configuration
Edit these settings to change which cohorts are matched and how strict matching is.

### Key parameters
- `COVARIATES`: columns used to estimate propensity scores (and therefore match on)
- `MATCH_RATIO`: number of controls per treated subject (1 means 1:1 matching)
- `CALIPER`: maximum allowed distance in propensity score (on *logit scale*, expressed as SDs); smaller is stricter
- `RANDOM_STATE`: controls reproducibility
- `REPLACE`: whether to allow matching the same control to multiple treated subjects (True = with replacement, False = without replacement)

In [ ]:
# Paths
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'UKB', 'cohorts')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'data', 'UKB', 'cohorts')
PLOTS_DIR = os.path.join(PROJECT_ROOT, 'reports', 'plots')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# Which depression cohort file to load
DEPRESSION_INC = ['F32']
DEPRESSION_TAG = '_'.join(DEPRESSION_INC)

CONTROL_FILE = 'control_cohort.csv'
TREATED_FILE = f'depression_cohort_{DEPRESSION_TAG}.csv'

# Matching setup
COVARIATES = ['p21003_i2', 'p31']  # age, sex (as used in cohort_matching_main.py)
TREATMENT_INDICATOR = 'depression_status'
MATCH_RATIO = 1
CALIPER = 0.2 # in standard deviations of the logit of the propensity score
RANDOM_STATE = 42
REPLACE = False

# Suffix for subject ID extraction
SUBJECT_ID_SUFFIX = '/i2'

# Output paths
MATCHED_CONTROLS_OUT = os.path.join(OUTPUT_DIR, f'matched_control_cohort_{DEPRESSION_TAG}.csv')
COMBINED_OUT = os.path.join(OUTPUT_DIR, f'combined_cohort_{DEPRESSION_TAG}.csv')
BALANCE_OUT = os.path.join(OUTPUT_DIR, f'balance_statistics_{DEPRESSION_TAG}.csv')
PLOT_OUT = os.path.join(PLOTS_DIR, f'propensity_score_distributions_{DEPRESSION_TAG}.svg')

## 3) Load cohorts
Loads the (unmatched) control cohort and the treated (depression) cohort created by the cohort selection step.
Both should contain the covariate columns in `COVARIATES`.

In [ ]:
control_path = os.path.join(DATA_DIR, CONTROL_FILE)
treated_path = os.path.join(DATA_DIR, TREATED_FILE)

control_cohort = pd.read_csv(control_path)
depression_cohort = pd.read_csv(treated_path)

print('Control cohort:', len(control_cohort))
print('Depression cohort:', len(depression_cohort))
print('Covariates:', COVARIATES)

missing_control = [c for c in COVARIATES if c not in control_cohort.columns]
missing_treated = [c for c in COVARIATES if c not in depression_cohort.columns]
print('Missing in control:', missing_control)
print('Missing in treated:', missing_treated)

## 4) Estimate propensity scores and match
`propensity_score_matching` does:
1. Concatenates control+treated
2. Preprocesses covariates (standardizes numeric, one-hot encodes categorical)
3. Fits a cross-validated logistic regression to estimate propensity scores
4. Matches controls to treated subjects using caliper-based nearest neighbors

If you set `return_propensity_scores=True`, it also returns the combined pre-matching table including `propensity_score` for diagnostics.

In [ ]:
matched_controls, combined_before = propensity_score_matching(
    control_df=control_cohort,
    treated_df=depression_cohort,
    covariates=COVARIATES,
    treatment_indicator=TREATMENT_INDICATOR,
    match_ratio=MATCH_RATIO,
    caliper=CALIPER,
    random_state=RANDOM_STATE,
    replace=REPLACE,
    return_propensity_scores=True,
)

print('Matched controls:', len(matched_controls))
display(matched_controls.head())

print('Combined (before matching) rows:', len(combined_before))
display(combined_before.head())

## 5) Assess covariate balance (SMD)
This computes standardized mean differences (SMD) for each covariate **before** and **after** matching.

In [ ]:
balance_stats = assess_covariate_balance(
    control_df=control_cohort,
    treated_df=depression_cohort,
    matched_control_df=matched_controls,
    covariates=COVARIATES,
)
display(balance_stats)

well_balanced_before = (balance_stats['smd_before'].abs() < 0.1).sum()
well_balanced_after = (balance_stats['smd_after'].abs() < 0.1).sum()
print(f'Well-balanced (|SMD|<0.1) before: {well_balanced_before}/{len(balance_stats)}')
print(f'Well-balanced (|SMD|<0.1) after:  {well_balanced_after}/{len(balance_stats)}')

## 6) Plot propensity score distributions
These diagnostics show overlap/common support between treated and controls. Poor overlap can mean you need different covariates, a different caliper, or a different matching approach.

In [ ]:
plot_propensity_distributions(
    propensity_scores=combined_before['propensity_score'],
    treatment=combined_before[TREATMENT_INDICATOR],
    save_path=PLOT_OUT,
)

## 7) Combine matched controls + treated
`combine_matched_cohorts` creates a single DataFrame for downstream analysis and adds `TREATMENT_INDICATOR` (0=control, 1=treated).
By default it drops matching-only columns like `treated_eid` and `propensity_score` from the matched controls.

In [ ]:
combined_cohort = combine_matched_cohorts(
    matched_control_df=matched_controls,
    treated_df=depression_cohort,
    treatment_column=TREATMENT_INDICATOR,
    drop_matching_cols=True,
)
display(combined_cohort.head())
print(combined_cohort[TREATMENT_INDICATOR].value_counts())

## 8) Extract subject IDs with suffixes
This step extracts subject IDs from the matched control cohort and saves them to a text file, appending a specified suffix to each ID. This suffix is the imaging instance we are interested in (initial imaging visit of all subjects).

In [ ]:
extract_subject_ids(
    cohort_df=matched_controls,
    eid_column='control_eid',
    suffix=SUBJECT_ID_SUFFIX,
    out_path=os.path.join(OUTPUT_DIR, 'control_subject_ids.txt')
)

## 9) Save outputs
This writes the matched controls, combined cohort, and balance statistics to disk.

In [ ]:
matched_controls.to_csv(MATCHED_CONTROLS_OUT, index=False)
combined_cohort.to_csv(COMBINED_OUT, index=False)
balance_stats.to_csv(BALANCE_OUT, index=False)

print('Saved matched controls ->', MATCHED_CONTROLS_OUT)
print('Saved combined cohort  ->', COMBINED_OUT)
print('Saved balance stats    ->', BALANCE_OUT)
print('Saved PS plot          ->', PLOT_OUT)

## 10) Quick matching summary
This reproduces the high-level summary printed in the script, but in notebook form.

In [ ]:
# Matching performance
if 'treated_eid' in matched_controls.columns:
    matches_per_treated = matched_controls.groupby('treated_eid').size()
    print('Matches per treated subject:')
    print('  Mean:', float(matches_per_treated.mean()))
    print('  Min :', int(matches_per_treated.min()))
    print('  Max :', int(matches_per_treated.max()))
else:
    print('No treated_eid column found (may have been dropped).')

# Balance
well_balanced_before = (balance_stats['smd_before'].abs() < 0.1).sum()
well_balanced_after = (balance_stats['smd_after'].abs() < 0.1).sum()
print(f'Well-balanced (|SMD|<0.1) before: {well_balanced_before}/{len(balance_stats)}')
print(f'Well-balanced (|SMD|<0.1) after : {well_balanced_after}/{len(balance_stats)}')

# Combined composition
print('Combined cohort composition:')
print('  Controls:', int((combined_cohort[TREATMENT_INDICATOR] == 0).sum()))
print('  Treated :', int((combined_cohort[TREATMENT_INDICATOR] == 1).sum()))
print('  Total   :', int(len(combined_cohort)))